# BONUS CONTENT


# Day 33 — Custom Transformers


## 1. Learning Objectives
By the end of this notebook, you will be able to:
- Understand the limitations of Scikit-learn's built-in transformers.
- Write your own Python classes that inherit from `BaseEstimator` and `TransformerMixin`.
- Execute arbitrary Pandas code (like custom math or outlier removal) natively inside a `Pipeline`.


## 2. Prerequisites
- Day 5 (Pipelines).
- Day 28 (Feature Engineering).


## 3. Concept: The Pipeline Limitation
In Day 28, we did Feature Engineering (e.g., `df['Square_Footage'] = df['Width'] * df['Length']`). 
We did this globally, *before* we passed the data into a Pipeline or Cross Validation. 

While this usually works for simple math, what if your Feature Engineering requires calculating the `mean()` of a column to subtract from another? If you do this globally, you cause Data Leakage (the mean calculation saw the test data).
To perfectly lock your custom Pandas code inside the Pipeline, you must build a **Custom Transformer**.


## 4. Concept: Duck Typing
Scikit-learn Pipelines don't care *what* object you put inside them, as long as the object quacks like a duck. 
Specifically, the object must have a `.fit()` method and a `.transform()` method. 
If we write our own Python class with those two methods, the `Pipeline` will gladly accept it and execute it!


## 5. Scikit-learn API
To ensure our custom class perfectly mimics Scikit-learn, we inherit from two base classes:
```python
from sklearn.base import BaseEstimator, TransformerMixin

class MyTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        # Do custom Pandas stuff here
        return X
```


## 6. Simple Example: The Math Transformer
Let's write a Custom Transformer that takes a DataFrame with `Width` and `Length`, and automatically returns a DataFrame with a brand new `Square_Footage` column.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

# 1. Define the Custom Class
class AreaEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
        
    def fit(self, X, y=None):
        # We don't need to learn any math from the data, so just return self
        return self
        
    def transform(self, X):
        # X will be passed in as a Pandas DataFrame or Numpy Array
        # We'll assume X is a DataFrame for simplicity
        X_new = X.copy()
        X_new['Square_Footage'] = X_new['Width'] * X_new['Length']
        return X_new

print('Custom Transformer Class defined!')


## 7. Code Walkthrough
- We created `AreaEngineer`.
- `fit()`: Required by Scikit-learn, but we don't need it. We just `return self`.
- `transform()`: We create a copy of the incoming data, do our arbitrary Pandas math (`Width * Length`), and return the modified DataFrame.


## 8. Experiment: Putting it in a Pipeline
Now let's prove that Scikit-learn accepts our custom class natively.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# Create some raw data
X_raw = pd.DataFrame({
    'Width': [10, 20, 15],
    'Length': [20, 40, 30]
})
y = np.array([200, 800, 450])

# Build a Pipeline containing our custom class!
custom_pipe = Pipeline([
    ('area_creator', AreaEngineer()),
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

# Run fit! The pipeline will automatically route the data through AreaEngineer.
custom_pipe.fit(X_raw, y)

print('Pipeline executed successfully using the Custom Transformer!')


> It worked! The Pipeline routed `X_raw` into `AreaEngineer.transform()`, which added the column. Then it routed that 3-column DataFrame into `StandardScaler.fit_transform()`, and finally into `LinearRegression.fit()`.


## 9. Prediction Exercise
Read the following code, but **DO NOT RUN IT YET**.


In [ ]:
class OutlierRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        # Drop rows where 'Price' is > 1000
        return X[X['Price'] <= 1000]


> **Question:** The engineer above wrote a custom transformer to delete outliers. If they put this inside a `Pipeline` and run `.fit(X, y)`, the pipeline will pass `X` into `transform()`, and `X` will lose 5 rows.
> What happens to `y` when it reaches the `LinearRegression.fit(X, y)` step at the end of the Pipeline?

**Think before running the next cell!**


In [ ]:
print('CRASH: ValueError: Found input variables with inconsistent numbers of samples.')
print('\nWhy? Scikit-learn Pipelines ONLY transform X. They NEVER transform y.')
print('If your Custom Transformer deletes 5 rows from X, y still has all of its original rows.')
print('When the algorithm tries to align X and y at the very end, the lengths don\'t match, and the code explodes.')


> **Rule:** You can NEVER drop rows inside a Scikit-learn Pipeline. Pipelines are strictly for modifying/adding/scaling columns.


## 10. Real-World Example
**Text Preprocessing**: When dealing with raw Tweets, you often need to remove emojis, strip URLs, and lowercase everything. There is no `StandardScaler` for this. Data scientists write custom `TweetCleaner(BaseEstimator, TransformerMixin)` classes that run Regex functions on the text, and place them as the very first step in their production NLP pipelines.


## 11. Coding Exercise
Write a custom transformer called `ThresholdBinarizer`. 
Its `__init__` should take a `threshold` argument (e.g., `threshold=5`). 
Its `transform` method should return a DataFrame where every value is converted to `1` if it is greater than the threshold, and `0` otherwise. 
Test it on a small DataFrame!


In [ ]:
# YOUR CODE HERE
class ThresholdBinarizer(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0):
        self.threshold = threshold
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        # Returns 1 if > threshold, else 0
        return (X > self.threshold).astype(int)

# Test it
test_df = pd.DataFrame({'A': [1, 5, 10], 'B': [-2, 7, 0]})
binarizer = ThresholdBinarizer(threshold=4)
print('Original:\n', test_df)
print('\nBinarized:\n', binarizer.fit_transform(test_df))


## 12. Summary of Bonus Day 33
- You can inject arbitrary Python/Pandas logic into Scikit-learn by inheriting from `BaseEstimator` and `TransformerMixin`.
- You must implement a `.fit()` method that returns `self`.
- You must implement a `.transform()` method that modifies and returns `X`.
- Custom Transformers cannot drop rows because they cannot modify `y`.
